# DeepSORVF — Phase 2: C3, C4, C5 (avec AIS)

Exécute les configurations **avec fusion AIS** : DTW, OAR, Binding, Hungarian.
Résultats sauvegardés dans `ablation_results/phase2/`.

**Indépendant de Phase 1** — peut être lancé en parallèle sur un 2ème compte Colab.

---
## 0. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Extract zip, copy clips and install dependencies
import os, zipfile, shutil, glob

ZIP_PATH = '/content/drive/MyDrive/DeepSORVF_Colab.zip'
DRIVE_CLIPS = '/content/drive/MyDrive/data/clips'
PROJECT_ROOT = '/content/DeepSORVF'

# Always re-extract to avoid stale code
if os.path.exists(PROJECT_ROOT):
    shutil.rmtree(PROJECT_ROOT)

print('Extracting zip...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(PROJECT_ROOT)

# Clear __pycache__
for p in glob.glob(os.path.join(PROJECT_ROOT, '**', '__pycache__'), recursive=True):
    shutil.rmtree(p)

# Copy clips from Drive
local_clips = os.path.join(PROJECT_ROOT, 'data', 'clips')
if os.path.isdir(DRIVE_CLIPS):
    print(f'Copying clips from Drive...')
    shutil.copytree(DRIVE_CLIPS, local_clips, dirs_exist_ok=True)
    print(f'Copied {len(os.listdir(local_clips))} clips')
else:
    print(f'ERROR: {DRIVE_CLIPS} not found! Upload data/clips/ to Drive root.')

In [ ]:
# Install dependencies
!pip install ultralytics==8.4.121 filterpy lap easydict geopy pyproj fastdtw loguru scikit-image

import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
# Verify clips and weights
clips_dir = os.path.join(PROJECT_ROOT, 'data', 'clips')
CLIPS = sorted([d for d in os.listdir(clips_dir)
                if os.path.isdir(os.path.join(clips_dir, d))])
print(f'Detected {len(CLIPS)} clips: {CLIPS}\n')

for clip in CLIPS:
    clip_path = os.path.join(clips_dir, clip)
    files = os.listdir(clip_path)
    video = [f for f in files if f.endswith(('.mp4', '.avi'))]
    ais_dir = os.path.join(clip_path, 'ais')
    ais_count = len(os.listdir(ais_dir)) if os.path.isdir(ais_dir) else 0
    print(f'  {clip}: video={video[0] if video else "MISSING"}, ais={ais_count}')

weights_dir = os.path.join(PROJECT_ROOT, 'weights')
print()
for name in ['best.pt', 'YOLOX-final.pth', 'ckpt.t7']:
    path = os.path.join(weights_dir, name)
    if os.path.exists(path):
        print(f'  {name}: {os.path.getsize(path)/1e6:.1f} MB')
    else:
        print(f'  {name}: MISSING')

---
## 1. Quick Test

In [ ]:
# Quick test: C5 on clip-02 (50 frames)
import sys
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from run_ablation import run_pipeline

print('=== Quick test: C5 / clip-02 (50 frames) ===')
stats = run_pipeline(
    clip_name='clip-02',
    result_dir=os.path.join(PROJECT_ROOT, 'quick_test'),
    config_name='C5',
    max_frames=50,
    use_ensemble=True, use_static_filter=True,
    ais_enabled=True, anti=1,
    use_dtw=True, use_angle_penalty=True,
    use_binding=True, use_hungarian=True,
)
print(f'Result: {stats}')

---
## 2. Phase 2 — C3, C4, C5 (avec AIS)
Skip les configs déjà traitées. Relancer sans problème.

In [ ]:
# Phase 2: C3, C4, C5 — skip existing, merge results
import json

PHASE2 = {
    'C3': dict(use_ensemble=True, use_static_filter=True, ais_enabled=True, anti=0),
    'C4': dict(use_ensemble=True, use_static_filter=True, ais_enabled=True, anti=1),
    'C5': dict(use_ensemble=True, use_static_filter=True, ais_enabled=True, anti=1),
}

RESULT_DIR = os.path.join(PROJECT_ROOT, 'ablation_results', 'phase2')
os.makedirs(RESULT_DIR, exist_ok=True)

# Load existing results
summary_path = os.path.join(RESULT_DIR, 'phase2_summary.json')
phase2_results = []
if os.path.exists(summary_path):
    with open(summary_path) as f:
        phase2_results = json.load(f)
    print(f'Loaded {len(phase2_results)} existing results')

done = set((r['clip'], r['config']) for r in phase2_results if 'error' not in r)

for clip in CLIPS:
    for config_name, flags in PHASE2.items():
        if (clip, config_name) in done:
            print(f'  SKIP {config_name} / {clip} (already done)')
            continue
        res_dir = os.path.join(RESULT_DIR, clip, config_name)
        print(f'\n--- {config_name} / {clip} ---')
        try:
            stats = run_pipeline(
                clip_name=clip,
                result_dir=res_dir,
                config_name=config_name,
                **flags
            )
            phase2_results.append(stats)
        except Exception as e:
            print(f'  ERROR: {e}')
            phase2_results.append({'config': config_name, 'clip': clip, 'error': str(e)})

with open(summary_path, 'w') as f:
    json.dump(phase2_results, f, indent=2)

print(f'\n=== Phase 2 total: {len(phase2_results)} runs ===')

---
## 3. Résultats

In [ ]:
# Print summary table
import json

RESULT_DIR = os.path.join(PROJECT_ROOT, 'ablation_results', 'phase2')
summary_path = os.path.join(RESULT_DIR, 'phase2_summary.json')

if os.path.exists(summary_path):
    with open(summary_path) as f:
        results = json.load(f)

    print(f"{'Config':<6} {'Clip':<12} {'Frames':>8} {'Det-Sec':>8} {'Time':>8} {'ms/frm':>8}")
    print('-' * 60)
    for r in results:
        if 'error' in r:
            print(f"{r['config']:<6} {r['clip']:<12} ERROR: {r['error']}")
        else:
            print(f"{r['config']:<6} {r['clip']:<12} {r['total_frames']:>8} {r['detection_seconds']:>8} {r['wall_time_s']:>7.1f}s {r['avg_ms_per_frame']:>7.1f}")
    print(f'\nTotal: {len(results)} runs')
else:
    print('No results yet')

---
## 4. Download Results

In [ ]:
# Zip and download phase2 results
import shutil
from google.colab import files

zip_path = '/tmp/phase2_results.zip'
shutil.make_archive('/tmp/phase2_results', 'zip',
                    os.path.join(PROJECT_ROOT, 'ablation_results', 'phase2'))
print(f'Zipped: {os.path.getsize(zip_path)/1e6:.1f} MB')
files.download(zip_path)